In [46]:
## imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import os
from pathlib import Path

# Detect if running on Colab
try:
    from google.colab import files
    IN_COLAB = True
    print("Running on Google Colab - models will be downloadable")
except:
    IN_COLAB = False
    print("Running locally")

# Define the models directory
NOTEBOOK_DIR = Path.cwd()
MODELS_DIR = NOTEBOOK_DIR / "models"

if IN_COLAB:
    # On Colab, create models dir in /content (Colab's working directory)
    MODELS_DIR = Path("/content/models")
    print(f"Models will be saved to Colab temp directory: {MODELS_DIR}")
    print("You'll be able to download them directly from Colab after training")
else:
    # Running locally
    print(f"Models will be saved to: {MODELS_DIR}")

Running on Google Colab - models will be downloadable
Models will be saved to Colab temp directory: /content/models
You'll be able to download them directly from Colab after training


In [47]:
# 1. Define the transformation
# VAEs often work well with pixel values scaled between 0 and 1, which ToTensor() handles automatically.
transform = transforms.Compose([
    transforms.ToTensor() 
])

# 2. Download and load the training dataset
train_dataset = torchvision.datasets.FashionMNIST(
    root='./data', 
    train=True,
    download=True, 
    transform=transform
)

# 3. Create the DataLoader
batch_size = 16
train_loader = torch.utils.data.DataLoader(
    train_dataset, 
    batch_size=batch_size,
    shuffle=True, 
    num_workers=2
)

# 4. Define the human-readable class names (Fashion-MNIST has 10 classes)
fashion_classes = (
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
)

print(f"Total training images: {len(train_dataset)}")

Total training images: 60000


In [48]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def reparameterize(mu, logvar):
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    return mu + eps * std

def vae_loss(recon_x, x, mu, logvar, beta=1.0):
    x = x.view(x.size(0), -1)
    recon_loss = F.binary_cross_entropy(recon_x, x, reduction='sum')
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + beta * kl, recon_loss, kl

Regularized MLP based VAE 

In [49]:
class VAE_MLP_Regularized(nn.Module):
    def __init__(self, latent_dim=32, dropout=0.1):
        super().__init__()
        
        # Encoder
        self.fc1 = nn.Linear(784, 512)
        self.ln1 = nn.LayerNorm(512)
        self.fc2 = nn.Linear(512, 256)
        self.ln2 = nn.LayerNorm(256)
        
        self.fc_mu = nn.Linear(256, latent_dim)
        self.fc_logvar = nn.Linear(256, latent_dim)
        
        self.dropout = nn.Dropout(dropout)
        
        # Decoder
        self.fc3 = nn.Linear(latent_dim, 256)
        self.ln3 = nn.LayerNorm(256)
        self.fc4 = nn.Linear(256, 512)
        self.ln4 = nn.LayerNorm(512)
        self.fc5 = nn.Linear(512, 784)

    def encode(self, x):
        x = x.view(x.size(0), -1)
        h = F.relu(self.ln1(self.fc1(x)))
        h = self.dropout(h)
        h = F.relu(self.ln2(self.fc2(h)))
        return self.fc_mu(h), self.fc_logvar(h)

    def decode(self, z):
        h = F.relu(self.ln3(self.fc3(z)))
        h = self.dropout(h)
        h = F.relu(self.ln4(self.fc4(h)))
        return torch.sigmoid(self.fc5(h))

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar

In [50]:
def download_model_from_colab(filename):
    """Helper function to download model from Colab to your local laptop"""
    if IN_COLAB:
        file_path = MODELS_DIR / filename
        if file_path.exists():
            print(f"\n✓ Model trained successfully!")
            print(f"Downloading {filename} to your local laptop...")
            files.download(str(file_path))
        else:
            print(f"Error: {filename} not found at {file_path}")
    else:
        print(f"Model saved locally at: {MODELS_DIR / filename}")

def train(model, train_loader, epochs=10, lr=1e-3, beta=1.0):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        total_loss = 0
        total_kl = 0
        total_recon = 0

        for x, _ in train_loader:
            x = x.to(device)

            recon, mu, logvar = model(x)
            loss, recon_loss, kl = vae_loss(recon, x, mu, logvar, beta)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            total_kl += kl.item()
            total_recon += recon_loss.item()

        print(f"Epoch {epoch+1}: Loss={total_loss:.2f}, Recon={total_recon:.2f}, KL={total_kl:.2f}")
    # save model
    MODELS_DIR.mkdir(exist_ok=True)
    save_path = MODELS_DIR / "vae_mlp_regularized.pth"
    torch.save(model.state_dict(), str(save_path))
    print(f"Model saved to: {save_path}")
    download_model_from_colab("vae_mlp_regularized.pth")

In [51]:
train(model=VAE_MLP_Regularized(), train_loader=train_loader, epochs=10, lr=1e-3, beta=1.0)

Epoch 1: Loss=16283133.06, Recon=15639026.76, KL=644106.31
Epoch 2: Loss=15125114.44, Recon=14449454.17, KL=675660.27
Epoch 3: Loss=14904135.16, Recon=14205720.74, KL=698414.42
Epoch 4: Loss=14788645.65, Recon=14075691.67, KL=712953.98
Epoch 5: Loss=14704918.73, Recon=13983267.61, KL=721651.12
Epoch 6: Loss=14642601.07, Recon=13915604.98, KL=726996.08
Epoch 7: Loss=14586925.80, Recon=13853491.70, KL=733434.10
Epoch 8: Loss=14542528.28, Recon=13805648.33, KL=736879.95
Epoch 9: Loss=14509496.08, Recon=13768260.57, KL=741235.51
Epoch 10: Loss=14483034.44, Recon=13737135.80, KL=745898.65
Model saved to: /content/models/vae_mlp_regularized.pth

✓ Model trained successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

CNN MODELS

In [52]:
def reparameterize(mu, logvar):
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    return mu + eps * std

def vae_loss(recon_x, x, mu, logvar, beta=1.0):
    recon_loss = F.binary_cross_entropy(recon_x, x, reduction="sum")
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    total = recon_loss + beta * kl
    return total, recon_loss, kl

In [53]:
class VAE_CNN_Baseline(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()

        # Encoder: 1x28x28 -> 64x7x7
        self.enc = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=4, stride=2, padding=1),   # 32x14x14
            nn.ReLU(),

            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),  # 64x7x7
            nn.ReLU()
        )

        self.fc_mu = nn.Linear(64 * 7 * 7, latent_dim)
        self.fc_logvar = nn.Linear(64 * 7 * 7, latent_dim)

        # Decoder
        self.fc_dec = nn.Linear(latent_dim, 64 * 7 * 7)

        self.dec = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1), # 32x14x14
            nn.ReLU(),

            nn.ConvTranspose2d(32, 1, kernel_size=4, stride=2, padding=1),  # 1x28x28
            nn.Sigmoid()
        )

    def encode(self, x):
        h = self.enc(x)
        h = h.view(x.size(0), -1)
        return self.fc_mu(h), self.fc_logvar(h)

    def decode(self, z):
        h = self.fc_dec(z)
        h = h.view(z.size(0), 64, 7, 7)
        return self.dec(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar

In [54]:
class VAE_CNN_Deep(nn.Module):
    def __init__(self, latent_dim=64):
        super().__init__()

        # Encoder: deeper stack
        self.enc = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=1, padding=1),   # 32x28x28
            nn.ReLU(),

            nn.Conv2d(32, 64, 4, stride=2, padding=1),  # 64x14x14
            nn.ReLU(),

            nn.Conv2d(64, 128, 4, stride=2, padding=1), # 128x7x7
            nn.ReLU(),

            nn.Conv2d(128, 128, 3, stride=1, padding=1), # 128x7x7
            nn.ReLU()
        )

        self.fc_mu = nn.Linear(128 * 7 * 7, latent_dim)
        self.fc_logvar = nn.Linear(128 * 7 * 7, latent_dim)

        # Decoder
        self.fc_dec = nn.Linear(latent_dim, 128 * 7 * 7)

        self.dec = nn.Sequential(
            nn.ConvTranspose2d(128, 128, 3, stride=1, padding=1),   # 128x7x7
            nn.ReLU(),

            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),    # 64x14x14
            nn.ReLU(),

            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),     # 32x28x28
            nn.ReLU(),

            nn.Conv2d(32, 1, kernel_size=3, stride=1, padding=1),
            nn.Sigmoid()
        )

    def encode(self, x):
        h = self.enc(x)
        h = h.view(x.size(0), -1)
        return self.fc_mu(h), self.fc_logvar(h)

    def decode(self, z):
        h = self.fc_dec(z)
        h = h.view(z.size(0), 128, 7, 7)
        return self.dec(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar

In [55]:
def train(model, train_loader, save_path, epochs=10, lr=1e-3, beta=1.0):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()

        total_loss = 0
        total_rec = 0
        total_kl = 0

        for x, _ in train_loader:
            x = x.to(device)

            recon, mu, logvar = model(x)

            loss, rec, kl = vae_loss(recon, x, mu, logvar, beta)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            total_rec += rec.item()
            total_kl += kl.item()

        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Loss: {total_loss:.2f} | "
            f"Recon: {total_rec:.2f} | "
            f"KL: {total_kl:.2f}"
        )
    # save model
    MODELS_DIR.mkdir(exist_ok=True)
    full_path = MODELS_DIR / save_path
    torch.save(model.state_dict(), str(full_path))
    print(f"Model saved to: {full_path}")
    download_model_from_colab(save_path)

In [56]:
# ---------------------------------------------------
# Usage
# ---------------------------------------------------

# Baseline CNN
model = VAE_CNN_Baseline(latent_dim=32)
train(model, train_loader, save_path="vae_cnn_baseline.pth", epochs=10, lr=1e-3, beta=1.0)

Epoch 1/10 | Loss: 15557165.39 | Recon: 14463019.79 | KL: 1094145.60
Epoch 2/10 | Loss: 14788239.32 | Recon: 13719310.20 | KL: 1068929.12
Epoch 3/10 | Loss: 14667984.18 | Recon: 13601257.34 | KL: 1066726.84
Epoch 4/10 | Loss: 14598513.08 | Recon: 13534308.23 | KL: 1064204.85
Epoch 5/10 | Loss: 14555492.28 | Recon: 13494353.74 | KL: 1061138.55
Epoch 6/10 | Loss: 14523850.72 | Recon: 13466012.55 | KL: 1057838.17
Epoch 7/10 | Loss: 14499953.32 | Recon: 13443240.90 | KL: 1056712.43
Epoch 8/10 | Loss: 14482297.35 | Recon: 13427305.82 | KL: 1054991.53
Epoch 9/10 | Loss: 14465524.45 | Recon: 13413224.09 | KL: 1052300.36
Epoch 10/10 | Loss: 14451447.26 | Recon: 13401254.26 | KL: 1050193.00
Model saved to: /content/models/vae_cnn_baseline.pth

✓ Model trained successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [57]:
# Deep CNN
model = VAE_CNN_Deep(latent_dim=64)
train(model, train_loader, save_path="vae_cnn_deep.pth", epochs=10, lr=1e-3, beta=1.0)

Epoch 1/10 | Loss: 15617440.46 | Recon: 14653439.85 | KL: 964000.60
Epoch 2/10 | Loss: 14707349.65 | Recon: 13722393.06 | KL: 984956.59
Epoch 3/10 | Loss: 14549100.46 | Recon: 13565912.80 | KL: 983187.67
Epoch 4/10 | Loss: 14460409.46 | Recon: 13473810.97 | KL: 986598.48
Epoch 5/10 | Loss: 14402083.09 | Recon: 13417637.07 | KL: 984446.03
Epoch 6/10 | Loss: 14365288.51 | Recon: 13379764.32 | KL: 985524.18
Epoch 7/10 | Loss: 14334902.89 | Recon: 13350679.21 | KL: 984223.68
Epoch 8/10 | Loss: 14312379.88 | Recon: 13330581.26 | KL: 981798.63
Epoch 9/10 | Loss: 14295873.53 | Recon: 13313495.56 | KL: 982377.98
Epoch 10/10 | Loss: 14280744.60 | Recon: 13296996.67 | KL: 983747.94
Model saved to: /content/models/vae_cnn_deep.pth

✓ Model trained successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>